# 🔬 Device Physics · 📐 Truss CAD · 🧮 Kids Calculus · 🧪 10k Experiments
## 👁️ Face Recognition · 🌀 Golden Ratio · 🔧 POSIX Makefile

> **NYU device physics summer + building instructions for minors**
> Every topic maps onto a physical system:
> electrons in a junction → forces in a truss → rates of change → symmetry.

| § | Topic | The question |
|---|---|---|
| §1 | Device physics | How does a transistor switch? |
| §2 | Calculus for kids (SymPy) | What IS a derivative? Show me pixels. |
| §3 | Symmetrical truss + CAD | How do bridges stand up? (SymPy linear system) |
| §4 | 10,000 experiments | How many trials until I know for sure? |
| §5 | Face recognition | How does a number become a face? |
| §6 | Golden ratio + buildings | Why do cathedrals feel right? |
| §7 | POSIX Makefile for Python | How do pros build projects? |

In [ ]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch, Rectangle, FancyBboxPatch
from matplotlib.lines import Line2D
from scipy import stats
from scipy.optimize import fsolve
import itertools, warnings, textwrap as tw
warnings.filterwarnings('ignore')
sp.init_printing(use_latex='mathjax')
from IPython.display import display, Math, Markdown

plt.rcParams.update({
    'figure.facecolor':'#f8f9fa','axes.facecolor':'#ffffff',
    'axes.edgecolor':'#333','text.color':'#111',
    'xtick.color':'#333','ytick.color':'#333',
    'axes.labelcolor':'#333','grid.color':'#ddd',
    'grid.alpha':0.7,'axes.grid':True,'figure.dpi':100,
    'lines.linewidth':2,'font.size':10,
})
np.random.seed(2026)
print('Libraries loaded. NYU Device Physics Summer — lets go.')

---
## §1 🔬 Device Physics — PN Junction · MOSFET · Band Diagrams

**PN junction** diode: thermal equilibrium → built-in potential $V_{bi}=\frac{kT}{q}\ln\frac{N_A N_D}{n_i^2}$

Depletion width: $W=\sqrt{\frac{2\varepsilon_s}{q}\left(\frac{1}{N_A}+\frac{1}{N_D}\right)(V_{bi}-V_A)}$

**Shockley diode equation:** $I = I_0\left(e^{qV/nkT}-1\right)$

**MOSFET:** threshold $V_T = V_{FB} + 2\phi_F + Q_{dep}/C_{ox}$

In saturation: $I_D = \frac{\mu_n C_{ox}}{2}\frac{W}{L}(V_{GS}-V_T)^2$

**Band diagram** = electrostatic potential energy of an electron through the device.
Fermi level $E_F$ is flat at equilibrium (no net current).

In [ ]:
# §1 — Device physics: PN junction + MOSFET

q   = 1.602e-19   # C
k_B = 1.381e-23   # J/K
T   = 300         # K
kT  = k_B*T/q    # V  (thermal voltage ≈ 0.026 V)
eps_Si = 11.7*8.854e-12  # F/m
ni_Si  = 1.5e16          # m^-3 (intrinsic carrier density at 300K)

print(f'§1 kT/q = {kT*1000:.2f} mV  (thermal voltage)')

# §1.1 PN junction
N_A = 1e23; N_D = 1e22  # m^-3
V_bi = kT * np.log(N_A*N_D/ni_Si**2)
print(f'§1.1 Built-in potential V_bi = {V_bi:.3f} V')

V_A_arr = np.linspace(-1.5, 0.65, 500)  # applied bias
W_dep   = np.sqrt(2*eps_Si/q * (1/N_A+1/N_D) * np.clip(V_bi-V_A_arr,0.01,20))
x_n     = W_dep * N_A/(N_A+N_D)
x_p     = W_dep * N_D/(N_A+N_D)

# Electric field peak at junction
E_peak  = -q*N_D*x_n/eps_Si
print(f'§1.1 E_peak (zero bias) = {np.abs(E_peak[V_A_arr<0.01][0])/1e6:.2f} MV/m')

# §1.2 Diode I-V (Shockley)
I0_fwd = 1e-12   # A  saturation current
n_id   = 1.0     # ideality
I_diode= I0_fwd*(np.exp(V_A_arr/(n_id*kT)) - 1)
# Limit at breakdown: simplified (no avalanche model, just clip)
V_BD   = -1.2
I_diode[V_A_arr < V_BD] = -1e-3*(1+np.exp(-10*(V_A_arr[V_A_arr<V_BD]-V_BD)))

# §1.3 MOSFET I-V
mu_n   = 0.04     # m^2/V/s  (electron mobility)
Cox    = 5e-3     # F/m^2  (SiO2, tox=7nm)
W_mos  = 10e-6   # m
L_mos  = 0.1e-6  # m  (100nm)
VT     = 0.5     # V threshold
phi_F  = kT*np.log(N_A/ni_Si)

VGS_arr = np.array([0.7,0.9,1.1,1.3,1.5,1.7])
VDS_arr = np.linspace(0,2.0,300)
beta    = mu_n*Cox*W_mos/L_mos

ID_all = []
for VGS in VGS_arr:
    VGS_eff = VGS - VT
    if VGS_eff <= 0:
        ID_all.append(np.zeros_like(VDS_arr))
        continue
    ID = np.where(VDS_arr <= VGS_eff,
                  beta*(VGS_eff*VDS_arr - VDS_arr**2/2),   # triode
                  beta*VGS_eff**2/2 * np.ones_like(VDS_arr)) # saturation
    ID_all.append(ID*1000)  # mA
ID_all = np.array(ID_all)

print(f'§1.3 MOSFET: β={beta*1000:.1f}mA/V², VT={VT}V, {W_mos*1e6:.0f}/{L_mos*1e9:.0f}um/nm')

# §1.4 Band diagram: p-n junction at zero bias
x_junction = np.linspace(-200e-9, 200e-9, 500)  # -200nm to +200nm
x_p_edge   = -x_p[250]; x_n_edge = x_n[250]

# Charge density
rho = np.where(x_junction < 0,
               np.where(x_junction > x_p_edge, -q*N_A, 0),
               np.where(x_junction < x_n_edge,  q*N_D, 0))

# Electric field (integrate rho)
E_field = np.cumsum(rho)*(x_junction[1]-x_junction[0])/eps_Si
E_field -= E_field.min()
E_field  = E_field - E_field.max()/2   # center

# Potential (integrate E)
phi_x = -np.cumsum(E_field)*(x_junction[1]-x_junction[0])
phi_x -= phi_x[0]   # reference

# Band edges
E_c = -phi_x              # conduction band (eV)
E_v = E_c - 1.12          # valence band (Si bandgap 1.12eV)
E_F = np.where(x_junction<0, -0.01, -1.11)   # Fermi level (flat at equilibrium)

# ── Plots ─────────────────────────────────────────────────────────
fig = plt.figure(figsize=(16,10))
gs  = gridspec.GridSpec(2,4,fig,hspace=0.4,wspace=0.35)

# Diode I-V
ax1 = fig.add_subplot(gs[0,0])
ax1.semilogy(V_A_arr[V_A_arr>-0.1], np.abs(I_diode[V_A_arr>-0.1])*1e6,
             'royalblue',lw=2.5,label='|I| (μA)')
ax1.set_xlabel('V_A (V)'); ax1.set_ylabel('|I| (μA, log)')
ax1.set_title('§1.2 Shockley diode I-V\n$I=I_0(e^{V/nV_T}-1)$')
ax1.axvline(0,color='gray',lw=0.5,ls='--')
ax1.set_xlim(-0.2,0.7)
ax2 = ax1.twinx()
ax2.plot(V_A_arr, I_diode*1e6,'royalblue',lw=1.5,alpha=0.5)
ax2.set_ylabel('I (μA, linear)',color='royalblue')

# MOSFET I-V
ax3 = fig.add_subplot(gs[0,1])
colors_mos = plt.cm.Blues(np.linspace(0.4,1.0,len(VGS_arr)))
for i,(VGS,ID) in enumerate(zip(VGS_arr,ID_all)):
    if VGS>VT:
        ax3.plot(VDS_arr,ID,color=colors_mos[i],lw=2,label=f'V_GS={VGS}V')
# Saturation boundary
VDS_sat = VGS_arr - VT
ID_sat  = beta*(VGS_arr-VT)**2/2*1000
ax3.plot(VDS_sat[VGS_arr>VT],ID_sat[VGS_arr>VT],'k--',lw=1.5,label='Sat boundary')
ax3.set_xlabel('V_DS (V)'); ax3.set_ylabel('I_D (mA)')
ax3.set_title(f'§1.3 NMOS I-V curves\nW/L={W_mos*1e6:.0f}/{L_mos*1e9:.0f}nm, VT={VT}V')
ax3.legend(fontsize=6,loc='upper left')

# Band diagram
ax4 = fig.add_subplot(gs[0,2])
ax4.plot(x_junction*1e9, E_c,'navy',lw=2.5,label='$E_C$')
ax4.plot(x_junction*1e9, E_v,'navy',lw=2.5,label='$E_V$')
ax4.fill_between(x_junction*1e9,E_v,E_c,alpha=0.05,color='navy')
ax4.axhline(-0.06,color='red',lw=2,ls='--',label='$E_F$ (flat at equil.)')
ax4.axvline(x_p_edge*1e9,color='gray',lw=0.7,ls=':')
ax4.axvline(x_n_edge*1e9,color='gray',lw=0.7,ls=':')
ax4.text(-80,-0.3,'p-type',ha='center',fontsize=10,color='gray')
ax4.text(80,-0.9,'n-type',ha='center',fontsize=10,color='gray')
ax4.set_xlabel('x (nm)'); ax4.set_ylabel('Energy (eV)')
ax4.set_title('§1.4 Band diagram: PN junction\n(equilibrium, zero bias)')
ax4.legend(fontsize=8)

# Depletion width vs bias
ax5 = fig.add_subplot(gs[0,3])
V_fwd = np.linspace(-2,0.6,300)
W_bias= np.sqrt(2*eps_Si/q*(1/N_A+1/N_D)*np.clip(V_bi-V_fwd,0.001,20))
ax5.plot(V_fwd,W_bias*1e9,'green',lw=2.5,label='W_dep (nm)')
ax5.axvline(0,color='gray',ls='--',lw=1,label='Zero bias')
ax5.axvline(V_bi,color='red',ls=':',lw=1.5,label=f'V_bi={V_bi:.2f}V')
ax5.set_xlabel('Applied bias V_A (V)'); ax5.set_ylabel('W_dep (nm)')
ax5.set_title('§1.1 Depletion width vs bias\n(reverse → wider, forward → narrows)')
ax5.legend(fontsize=8)

# MOSFET cross-section schematic
ax6 = fig.add_subplot(gs[1,:2])
ax6.set_xlim(0,10); ax6.set_ylim(-1,4); ax6.axis('off')
ax6.set_title('§1.3 MOSFET cross-section schematic', fontsize=11)
# p-substrate
ax6.add_patch(Rectangle((0,-1),10,2.5,facecolor='#e8f5e9',edgecolor='green',lw=1.5))
ax6.text(5,-0.5,'p-substrate (Si)',ha='center',va='center',fontsize=10,color='green')
# n+ source / drain
ax6.add_patch(Rectangle((0.5,0.5),1.5,1.0,facecolor='#1565c0',edgecolor='navy',lw=1.5))
ax6.text(1.25,1.0,'n⁺\nSource',ha='center',va='center',fontsize=8,color='white')
ax6.add_patch(Rectangle((8.0,0.5),1.5,1.0,facecolor='#1565c0',edgecolor='navy',lw=1.5))
ax6.text(8.75,1.0,'n⁺\nDrain',ha='center',va='center',fontsize=8,color='white')
# Gate oxide
ax6.add_patch(Rectangle((2.0,1.5),6.0,0.3,facecolor='#fff9c4',edgecolor='orange',lw=1.5))
ax6.text(5,1.65,'SiO₂ (gate oxide, ~7nm)',ha='center',va='center',fontsize=8,color='orange')
# Poly gate
ax6.add_patch(Rectangle((2.0,1.8),6.0,0.8,facecolor='#bdbdbd',edgecolor='#555',lw=1.5))
ax6.text(5,2.2,'Poly-Si Gate',ha='center',va='center',fontsize=10,fontweight='bold')
# Channel
ax6.add_patch(Rectangle((2.0,0.5),6.0,1.0,facecolor='#fff3e0',edgecolor='orange',lw=1,ls='--'))
ax6.text(5,1.0,'Channel\nL=100nm',ha='center',va='center',fontsize=8,color='darkorange')
# Arrows: V_GS, V_DS
ax6.annotate('',xy=(5,2.6),xytext=(5,3.4),
             arrowprops=dict(arrowstyle='<-',color='red',lw=2))
ax6.text(5.2,3.0,'$V_{GS}$',fontsize=12,color='red')
ax6.annotate('',xy=(8.75,2.0),xytext=(8.75,3.0),
             arrowprops=dict(arrowstyle='<-',color='blue',lw=2))
ax6.text(9.0,2.5,'$V_{DS}$',fontsize=12,color='blue')
ax6.text(5,3.6,f'β={beta*1000:.1f}mA/V²  |  V_T={VT}V  |  μ_n={mu_n*1e4:.0f}cm²/Vs',
         ha='center',fontsize=9,color='gray')

# Electric field in depletion region
ax7 = fig.add_subplot(gs[1,2:])
ax7.plot(x_junction*1e9, E_field/1e6,'#c62828',lw=2.5,label='E(x) MV/m')
ax7.fill_between(x_junction*1e9,0,E_field/1e6,alpha=0.15,color='red')
ax7_b = ax7.twinx()
ax7_b.plot(x_junction*1e9,phi_x,'royalblue',lw=2,ls='--',label='φ(x) V')
ax7_b.set_ylabel('φ(x) (V)',color='royalblue')
ax7.set_xlabel('x (nm)'); ax7.set_ylabel('E-field (MV/m)',color='#c62828')
ax7.set_title('§1.1 PN junction: E-field + potential φ(x)\n(built-in field opposes forward bias)')
ax7.axvline(0,color='gray',ls=':',lw=1)
ax7.legend(fontsize=8,loc='upper left')
ax7_b.legend(fontsize=8,loc='upper right')
plt.suptitle('§1: Device Physics — PN Junction · MOSFET I-V · Band Diagram',
             fontsize=12,fontweight='bold')
plt.savefig('tmp.png',bbox_inches='tight'); plt.show()

---
## §2 🧮 Calculus for Kids — SymPy Makes It Visual

**A derivative is a slope** — how fast is something changing RIGHT NOW?

If $f(x) = x^2$, then $f'(x) = 2x$ — the slope at $x=3$ is $6$.

**An integral is area** — add up infinitely many infinitely thin slices.

$\int_0^3 x^2\,dx = \left[\frac{x^3}{3}\right]_0^3 = 9$

**SymPy lets you check every step symbolically** — no guessing.

> If you can understand $f'(x) = \lim_{h\to 0}\frac{f(x+h)-f(x)}{h}$,
> you understand the entire semester.

In [ ]:
# §2 — Calculus for kids with SymPy

x, h, t, a = sp.symbols('x h t a', real=True)

# §2.1 Derivative from FIRST PRINCIPLES — show the limit
f_demo = x**3 - 2*x + 1
diff_quot = (f_demo.subs(x, x+h) - f_demo) / h
diff_quot_expand = sp.expand(diff_quot)
deriv_limit = sp.limit(diff_quot, h, 0)
print('§2.1 Derivative from first principles:')
print(f'  f(x) = {f_demo}')
print(f'  [f(x+h)-f(x)]/h = {diff_quot_expand}')
print(f'  lim h→0 = {deriv_limit}')
print(f'  sp.diff confirms: {sp.diff(f_demo,x)}')
display(Math(r'\lim_{h\to 0}\frac{(x+h)^3-2(x+h)+1 - (x^3-2x+1)}{h} = 3x^2-2'))

# §2.2 Gallery of derivatives — rules
print('\n§2.2 Derivative rules gallery:')
funcs = {
    'x^5':         x**5,
    'sin(x)':      sp.sin(x),
    'e^x':         sp.exp(x),
    'ln(x)':       sp.ln(x),
    'x^2*sin(x)':  x**2*sp.sin(x),
    'sin(x^2)':    sp.sin(x**2),
    '1/(1+x^2)':   1/(1+x**2),
    'x^x':         sp.exp(x*sp.ln(x)),
}
for name, f in funcs.items():    # loop: differentiate each
    df = sp.diff(f, x)
    print(f'  d/dx [{name}] = {sp.simplify(df)}')

# §2.3 Integral = area — visualize Riemann sum converging
f_area = sp.Lambda(x, x**2)
exact_area = sp.integrate(f_area(x), (x, 0, 3))
print(f'\n§2.3 Exact area ∫₀³ x² dx = {exact_area}')

x_cont = np.linspace(0,3,400)
y_cont = x_cont**2

# Riemann sums with different n
def riemann_sum(f, a, b, n, method='mid'):
    dx_r = (b-a)/n
    if method=='left':  xi = a + dx_r*np.arange(n)
    elif method=='right': xi = a + dx_r*(np.arange(n)+1)
    else:               xi = a + dx_r*(np.arange(n)+0.5)  # midpoint
    return dx_r * f(xi).sum(), xi, f(xi), dx_r

for n in [4,8,16,64,256]:    # loop: Riemann convergence
    approx,_,_,_ = riemann_sum(lambda x: x**2, 0, 3, n)
    print(f'  n={n:4d} rectangles → area ≈ {approx:.6f}  (error {abs(approx-9):.2e})')

# §2.4 Taylor series — visualizing convergence
x_ts = np.linspace(-np.pi,np.pi,400)
sin_true = np.sin(x_ts)
print('\n§2.4 Taylor series of sin(x):')
terms = []
approx_ts = np.zeros_like(x_ts)
for k in range(8):           # loop: add Taylor terms
    n_term = 2*k+1
    coeff  = (-1)**k / float(sp.factorial(n_term))
    approx_ts = approx_ts + coeff*x_ts**n_term
    err = np.max(np.abs(approx_ts - sin_true))
    terms.append((n_term,approx_ts.copy(),err))
    print(f'  Order {n_term}: max error = {err:.2e}')

# §2.5 Chain rule visualization
print('\n§2.5 Chain rule: d/dx[sin(x²)] = cos(x²)·2x')
inner = x**2; outer = sp.sin(x)
chain = sp.diff(sp.sin(x**2), x)
display(Math(r'\frac{d}{dx}\sin(x^2) = \cos(x^2)\cdot 2x'))

# ── Plots ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(2,3,figsize=(16,9))

# Derivative as slope
x_arr = np.linspace(-2,4,400)
f_arr = x_arr**3 - 2*x_arr + 1
x0_demo=2.0; f0=x0_demo**3-2*x0_demo+1; slope=3*x0_demo**2-2
tang_x = np.linspace(x0_demo-1.5,x0_demo+1.5,100)
tang_y = f0 + slope*(tang_x-x0_demo)
axes[0][0].plot(x_arr,f_arr,'royalblue',lw=2.5,label='f(x)=x³-2x+1')
axes[0][0].plot(tang_x,tang_y,'red',lw=2,ls='--',label=f"f'({x0_demo})={slope} (slope)")
axes[0][0].plot(x0_demo,f0,'ro',ms=10,zorder=5)
axes[0][0].set_xlim(-2,4); axes[0][0].set_ylim(-5,12)
axes[0][0].set_title('§2.1 Derivative = slope of tangent\nd/dx[x³-2x+1] = 3x²-2')
axes[0][0].legend(fontsize=8)

# Riemann sum (n=8)
approx8,xi8,yi8,dx8 = riemann_sum(lambda x: x**2, 0, 3, 8)
axes[0][1].fill_between(x_cont, 0, y_cont, alpha=0.15, color='royalblue', label='True area=9')
for i in range(len(xi8)):    # loop: draw rectangles
    axes[0][1].add_patch(Rectangle((xi8[i]-dx8/2,0),dx8,yi8[i],
                                    facecolor='orange',edgecolor='darkorange',alpha=0.6))
axes[0][1].plot(x_cont,y_cont,'royalblue',lw=2)
axes[0][1].set_title(f'§2.3 Riemann sum (n=8): area≈{approx8:.3f}\n∫₀³ x² dx = 9 exactly')
axes[0][1].set_xlabel('x'); axes[0][1].set_ylabel('f(x)=x²')
axes[0][1].legend(fontsize=8)

# Taylor series convergence
colors_t = plt.cm.RdYlGn(np.linspace(0.1,0.9,len(terms)))
axes[0][2].plot(x_ts,sin_true,'k',lw=3,label='sin(x) exact')
for (n_t,approx_t,err_t),col in zip(terms[::2],colors_t[::2]):  # loop: every other order
    axes[0][2].plot(x_ts,np.clip(approx_t,-2,2),color=col,lw=1.5,
                    alpha=0.8,label=f'order {n_t}')
axes[0][2].set_ylim(-1.5,1.5); axes[0][2].set_xlabel('x')
axes[0][2].set_title('§2.4 Taylor series: sin(x)\nmore terms = more accurate')
axes[0][2].legend(fontsize=7)

# Derivative gallery (text plot)
axes[1][0].axis('off')
table_text = ''.join([f'd/dx[{n}]\n  = {sp.simplify(sp.diff(f,x))}\n\n' for n,f in list(funcs.items())[:6]])
axes[1][0].text(0.02,0.98,table_text,transform=axes[1][0].transAxes,
                fontsize=9,verticalalignment='top',fontfamily='monospace',
                bbox=dict(boxstyle='round',facecolor='#f0f8ff',alpha=0.9))
axes[1][0].set_title('§2.2 Derivative rules\n(SymPy computes all)')

# Error vs n_rectangles
n_arr = np.arange(1,201)
errors_rs = [abs(riemann_sum(lambda x:x**2,0,3,n,'mid')[0]-9) for n in n_arr]  # loop
axes[1][1].loglog(n_arr,errors_rs,'royalblue',lw=2.5,label='Midpoint Riemann error')
slope_ref = errors_rs[0]*n_arr[0]**2/n_arr**2
axes[1][1].loglog(n_arr,slope_ref,'r--',lw=1.5,label='1/n² reference')
axes[1][1].set_xlabel('Number of rectangles n')
axes[1][1].set_ylabel('|Area error|')
axes[1][1].set_title('§2.3 Riemann convergence rate\n1/n² for midpoint rule')
axes[1][1].legend(fontsize=8)

# Chain rule visualization
x_ch = np.linspace(-2,2,400)
f_ch = np.sin(x_ch**2)
df_ch= np.cos(x_ch**2)*2*x_ch
axes[1][2].plot(x_ch,f_ch,'royalblue',lw=2.5,label='sin(x²)')
axes[1][2].plot(x_ch,df_ch,'red',lw=2,ls='--',label="d/dx sin(x²) = cos(x²)·2x")
axes[1][2].axhline(0,color='gray',lw=0.5)
axes[1][2].set_xlabel('x')
axes[1][2].set_title('§2.5 Chain rule: d/dx[sin(x²)]\nderivative is zero at extrema')
axes[1][2].legend(fontsize=8)
plt.suptitle('§2: Calculus for Kids — SymPy First Principles · Riemann · Taylor · Chain Rule',
             fontsize=12,fontweight='bold')
plt.tight_layout(); plt.show()

---
## §3 📐 Symmetrical Truss System + CAD Drawing

**Method of joints:** sum of forces at each free joint = 0.
For a truss with $j$ joints and $m$ members + $r$ reactions:
- $2j = m + r$ (determinate if equality holds)

**Pratt truss** (vertical tension, diagonal compression under gravity load):
diagonals slope inward toward center → diagonals in **tension**, verticals in **compression**.

**Symmetry halves the work:** solve half the truss, mirror the rest.

**Building instructions for a symmetrical Pratt truss:**
1. Lay top chord (compression, C-shape steel)
2. Bottom chord (tension, flat bar)
3. Verticals (short compression posts)
4. Diagonals (long tension cables or L-angles)
5. Connections: gusset plates at each joint

In [ ]:
# §3 — Symmetrical Pratt truss analysis + CAD

# Geometry: 6-panel Pratt truss, span 12m, height 2m
n_panels = 6; span = 12.0; H_truss = 2.0
panel_w  = span/n_panels    # 2m per panel

# Joint coordinates (bottom chord then top chord)
# Bottom joints: B0...B6 (y=0)
# Top joints:    T0...T6 (y=H_truss)
joints = {}
for i in range(n_panels+1):
    joints[f'B{i}'] = np.array([i*panel_w, 0.0])
    joints[f'T{i}'] = np.array([i*panel_w, H_truss])

joint_names = list(joints.keys())
J = len(joint_names)
jmap = {name:i for i,name in enumerate(joint_names)}

# Members: top chord, bottom chord, verticals, diagonals (Pratt: diagonals slope inward)
members = []
# Top chord: T0-T1, T1-T2, ..., T5-T6
for i in range(n_panels):
    members.append((f'T{i}',f'T{i+1}'))
# Bottom chord: B0-B1, ..., B5-B6
for i in range(n_panels):
    members.append((f'B{i}',f'B{i+1}'))
# Verticals: T0-B0 (support), T1-B1, ..., T6-B6
for i in range(n_panels+1):
    members.append((f'T{i}',f'B{i}'))
# Pratt diagonals: slope inward from both ends toward center
# Left half: B1-T2, B2-T3 (slope right-upward = tension under downward load)
# Right half: B5-T4, B4-T3 (slope left-upward = tension)
diagonals_left  = [(f'B{i}',f'T{i+1}') for i in range(1,n_panels//2)]
diagonals_right = [(f'B{n_panels-i}',f'T{n_panels-i-1}') for i in range(1,n_panels//2)]
members += diagonals_left + diagonals_right

M = len(members)
print(f'§3 Pratt truss: {J} joints, {M} members')
print(f'   2j={2*J}, m+r={M}+3={M+3} (need m+r=2j for determinacy → {2*J==M+3})')

# Reactions: pin at B0 (Rx0, Ry0), roller at B6 (Ry6)
# Loads: uniform distributed → point loads at each bottom joint
w_UDL  = 50.0   # kN per meter (total 600 kN on 12m span)
P_joint= w_UDL*panel_w    # 100 kN per joint
# Symmetric reactions
R_B0 = w_UDL*span/2; R_B6 = R_B0
print(f'   UDL={w_UDL}kN/m, panel load={P_joint}kN, reactions={R_B0}kN each')

# Build stiffness matrix using direct stiffness method (2D)
# Global DOF: 2 per joint = [ux,uy]*J
# Apply boundary conditions: B0 pinned (DOF 0,1 fixed), B6 roller (DOF 12 fixed)
n_dof = 2*J
K_glob = np.zeros((n_dof,n_dof))

def member_stiffness(name_i, name_j, EA=70000.0):
    pi = joints[name_i]; pj = joints[name_j]
    dx = pj[0]-pi[0]; dy = pj[1]-pi[1]
    L  = np.sqrt(dx**2+dy**2)
    c  = dx/L; s  = dy/L
    k  = EA/L * np.array([[c*c,c*s,-c*c,-c*s],
                            [c*s,s*s,-c*s,-s*s],
                            [-c*c,-c*s,c*c,c*s],
                            [-c*s,-s*s,c*s,s*s]])
    return k, L

for (ni,nj) in members:
    k,_ = member_stiffness(ni,nj)
    dofs = [2*jmap[ni],2*jmap[ni]+1,2*jmap[nj],2*jmap[nj]+1]
    for a,da in enumerate(dofs):
        for b,db in enumerate(dofs):
            K_glob[da,db] += k[a,b]

# Load vector
F_glob = np.zeros(n_dof)
for i in range(1,n_panels):   # loop: apply loads at bottom joints (not at supports)
    dof_y = 2*jmap[f'B{i}']+1
    F_glob[dof_y] = -P_joint

# Boundary conditions: pin at B0 → DOF 0,1=0; roller at B6 → DOF 12=0
fixed_dofs = [0,1,2*jmap['B6']]
free_dofs  = [d for d in range(n_dof) if d not in fixed_dofs]

K_ff = K_glob[np.ix_(free_dofs,free_dofs)]
F_f  = F_glob[free_dofs]
U_f  = np.linalg.solve(K_ff, F_f)

U_glob = np.zeros(n_dof)
for i,d in enumerate(free_dofs):
    U_glob[d] = U_f[i]

# Member forces
member_forces = {}
for (ni,nj) in members:
    k,L = member_stiffness(ni,nj)
    pi = joints[ni]; pj = joints[nj]
    dx=pj[0]-pi[0]; dy=pj[1]-pi[1]; LL=np.sqrt(dx**2+dy**2)
    c=dx/LL; s=dy/LL; EA=70000.0
    dofs=[2*jmap[ni],2*jmap[ni]+1,2*jmap[nj],2*jmap[nj]+1]
    u_local=[U_glob[d] for d in dofs]
    F_axial = EA/LL*(c*(u_local[2]-u_local[0])+s*(u_local[3]-u_local[1]))
    member_forces[(ni,nj)] = F_axial

print('\n§3 Member forces (kN, +tension, -compression):')
top_chord = [(f'T{i}',f'T{i+1}') for i in range(n_panels)]
bot_chord = [(f'B{i}',f'B{i+1}') for i in range(n_panels)]
for m in top_chord[:3]:
    print(f'  Top chord {m[0]}-{m[1]}: {member_forces[m]:.1f} kN')
for m in bot_chord[:3]:
    print(f'  Bot chord {m[0]}-{m[1]}: {member_forces[m]:.1f} kN')
for m in diagonals_left:
    print(f'  Diagonal  {m[0]}-{m[1]}: {member_forces[m]:.1f} kN ({"T" if member_forces[m]>0 else "C"})')

# ── CAD-style truss drawing ──
fig, axes = plt.subplots(2,2,figsize=(16,10))

# Main truss CAD view
ax_cad = axes[0][0]
ax_cad.set_xlim(-1,13); ax_cad.set_ylim(-1,4)
ax_cad.set_aspect('equal'); ax_cad.grid(True,alpha=0.3)
ax_cad.set_xlabel('x (m)'); ax_cad.set_ylabel('y (m)')
ax_cad.set_title('§3 Pratt Truss CAD — Symmetrical 6-Panel\nw=50kN/m, span=12m, H=2m')

# Draw members colored by force
F_vals = [member_forces[m] for m in members]
F_max  = max(abs(f) for f in F_vals)
for (ni,nj),F in zip(members,F_vals):
    pi=joints[ni]; pj=joints[nj]
    color = 'royalblue' if F > 0 else 'red'     # blue=tension, red=compression
    lw    = 3.0 + 3.0*abs(F)/F_max
    ax_cad.plot([pi[0],pj[0]],[pi[1],pj[1]],color=color,lw=lw,solid_capstyle='round',zorder=2)

# Draw joints
for name,pos in joints.items():
    ax_cad.plot(*pos,'ko',ms=8,zorder=3)
    ax_cad.text(pos[0],pos[1]+0.12,name,ha='center',va='bottom',fontsize=7)

# Support symbols
ax_cad.add_patch(mpatches.RegularPolygon(joints['B0']-np.array([0,0.3]),3,radius=0.2,
                                          color='green',zorder=4))
ax_cad.add_patch(mpatches.RegularPolygon(joints['B6']-np.array([0,0.3]),3,radius=0.2,
                                          color='green',zorder=4))
ax_cad.text(0,-0.6,'PIN',ha='center',fontsize=8,color='green')
ax_cad.text(12,-0.6,'ROLLER',ha='center',fontsize=8,color='green')

# Load arrows
for i in range(1,n_panels):
    bj=joints[f'B{i}']
    ax_cad.annotate('',xy=(bj[0],bj[1]),xytext=(bj[0],bj[1]+0.7),
                    arrowprops=dict(arrowstyle='-|>',color='orange',lw=2))
    ax_cad.text(bj[0]+0.1,bj[1]+0.4,f'{P_joint:.0f}kN',fontsize=7,color='darkorange')

# Legend
ax_cad.plot([],[],color='royalblue',lw=3,label='Tension (+)')
ax_cad.plot([],[],color='red',lw=3,label='Compression (-)')
ax_cad.legend(fontsize=9,loc='upper right')
ax_cad.text(6,3.5,f'Reactions: {R_B0:.0f}kN each (symmetric)',
            ha='center',fontsize=10,color='green')

# Force diagram (bar chart)
ax_fc = axes[0][1]
names_m   = [f'{a[0]}-{a[1]}' for a in members]
forces_m  = [member_forces[m] for m in members]
colors_bar= ['royalblue' if f>0 else 'red' for f in forces_m]
ax_fc.barh(range(len(forces_m)),forces_m,color=colors_bar,alpha=0.8,edgecolor='#333')
ax_fc.set_yticks(range(len(names_m)))
ax_fc.set_yticklabels(names_m,fontsize=7)
ax_fc.axvline(0,color='black',lw=1.5)
ax_fc.set_xlabel('Axial force (kN)  +tension / -compression')
ax_fc.set_title('§3 Member forces\n(Blue=tension, Red=compression)')

# Deflected shape
ax_def = axes[1][0]
scale  = 200   # exaggerate deformation
for (ni,nj) in members:
    pi=joints[ni]; pj=joints[nj]
    ui=[U_glob[2*jmap[ni]],U_glob[2*jmap[ni]+1]]
    uj=[U_glob[2*jmap[nj]],U_glob[2*jmap[nj]+1]]
    # Original
    ax_def.plot([pi[0],pj[0]],[pi[1],pj[1]],'lightgray',lw=1,ls='--',zorder=1)
    # Deflected
    ax_def.plot([pi[0]+scale*ui[0],pj[0]+scale*uj[0]],
                [pi[1]+scale*ui[1],pj[1]+scale*uj[1]],'royalblue',lw=2,zorder=2)
ax_def.set_xlim(-1,13); ax_def.set_ylim(-1.5,3.5); ax_def.set_aspect('equal')
ax_def.set_title(f'§3 Deflected shape (×{scale} exaggerated)\nMax bottom chord sag visible')
ax_def.set_xlabel('x (m)'); ax_def.set_ylabel('y (m)')

# Influence line for mid-span bottom chord
ax_il = axes[1][1]
member_il = ('B2','B3')   # center bottom chord
forces_il = []
positions  = np.linspace(0,12,13)
for x_load in positions:     # loop: move unit load
    # Find nearest bottom joint
    nearest = min(range(1,n_panels),key=lambda i: abs(joints[f'B{i}'][0]-x_load))
    F_test  = np.zeros(n_dof)
    F_test[2*jmap[f'B{nearest}']+1] = -1.0   # unit load
    try:
        U_test_f = np.linalg.solve(K_ff, F_test[free_dofs])
        U_test   = np.zeros(n_dof)
        for i,d in enumerate(free_dofs): U_test[d]=U_test_f[i]
        pi2=joints[member_il[0]]; pj2=joints[member_il[1]]
        dx2=pj2[0]-pi2[0]; dy2=pj2[1]-pi2[1]; LL2=np.sqrt(dx2**2+dy2**2)
        c2=dx2/LL2; s2=dy2/LL2; EA2=70000.0
        dofs2=[2*jmap[member_il[0]],2*jmap[member_il[0]]+1,
               2*jmap[member_il[1]],2*jmap[member_il[1]]+1]
        ul=[U_test[d] for d in dofs2]
        FA=EA2/LL2*(c2*(ul[2]-ul[0])+s2*(ul[3]-ul[1]))
        forces_il.append(FA)
    except: forces_il.append(0)
ax_il.plot(positions,forces_il,'royalblue',lw=2.5,marker='o',ms=6)
ax_il.axhline(0,color='black',lw=1)
ax_il.fill_between(positions,0,forces_il,alpha=0.2,color='royalblue')
ax_il.set_xlabel('Unit load position (m)'); ax_il.set_ylabel('Force in B2-B3 (kN/kN)')
ax_il.set_title('§3 Influence line: member B2-B3\n(tension when load anywhere on span)')
plt.suptitle('§3: Symmetrical Pratt Truss — Direct Stiffness FEM + CAD',
             fontsize=12,fontweight='bold')
plt.tight_layout(); plt.show()

---
## §4 🧪 10,000 Planned Experiments + §5 👁️ Face Recognition

**Statistical power** of an experiment:
$\text{Power} = P(\text{reject } H_0 \mid H_1 \text{ true}) = 1-\beta$

Required sample size for two-sample t-test:
$$n = \frac{2(z_{\alpha/2}+z_\beta)^2\sigma^2}{\delta^2}$$

With 10,000 subjects: detect effect size $\delta/\sigma \approx 0.04$ at 80% power.

**Face recognition pipeline:**
1. **Detect** — find face bbox (MTCNN / Viola-Jones / YOLO)
2. **Align** — warp to canonical pose using eye landmarks
3. **Embed** — 128-D or 512-D face embedding (FaceNet / ArcFace)
4. **Match** — cosine similarity or L2 distance in embedding space

Cosine similarity: $\cos\theta = \frac{\mathbf{e}_1\cdot\mathbf{e}_2}{\|\mathbf{e}_1\|\|\mathbf{e}_2\|}$
Threshold ~0.6 for same identity (dataset-dependent).

In [ ]:
# §4 — 10,000 experiments + §5 face recognition pipeline

# §4.1 Statistical power analysis
from scipy.stats import norm, t as t_dist, ttest_ind

alpha_level = 0.05
power_target= 0.80
z_alpha2    = norm.ppf(1-alpha_level/2)   # 1.96
z_beta      = norm.ppf(power_target)      # 0.84
print(f'§4.1 z_α/2={z_alpha2:.3f}, z_β={z_beta:.3f}')

# Sample size vs effect size
delta_over_sigma = np.linspace(0.02,1.0,200)
n_required       = 2*(z_alpha2+z_beta)**2 / delta_over_sigma**2
print(f'     At δ/σ=0.2 (small): n={int(2*(z_alpha2+z_beta)**2/0.2**2)} per group')
print(f'     At δ/σ=0.5 (medium): n={int(2*(z_alpha2+z_beta)**2/0.5**2)} per group')
print(f'     At δ/σ=0.8 (large): n={int(2*(z_alpha2+z_beta)**2/0.8**2)} per group')
print(f'     N=10000 detects δ/σ ≥ {np.sqrt(2*(z_alpha2+z_beta)**2/5000):.4f}')

# §4.2 Monte Carlo power simulation: 10,000 simulated experiments
n_sims   = 10000
n_per_grp= 50
effect_sz= 0.3    # in sigma units
sigma_exp= 1.0
rejections = 0
p_values   = []
for _ in range(n_sims):    # loop: simulate experiments
    ctrl  = np.random.normal(0, sigma_exp, n_per_grp)
    treat = np.random.normal(effect_sz*sigma_exp, sigma_exp, n_per_grp)
    _,p   = ttest_ind(ctrl, treat)
    p_values.append(p)
    if p < alpha_level: rejections += 1

empirical_power = rejections/n_sims
analytic_power  = 1 - t_dist.cdf(t_dist.ppf(1-alpha_level/2, df=2*(n_per_grp-1)),
                                   df=2*(n_per_grp-1),
                                   nc=effect_sz*sigma_exp/(sigma_exp*np.sqrt(2/n_per_grp)))
print(f'\n§4.2 MC simulation (n_sims={n_sims}, n={n_per_grp}/group, δ/σ={effect_sz}):')
print(f'     Empirical power = {empirical_power:.3f}')
print(f'     Analytic power  = {analytic_power:.3f}')

# §4.3 Type I / Type II error tradeoff
alpha_arr  = np.linspace(0.001,0.20,200)
# Effect size 0.3, n=50
se_diff    = sigma_exp*np.sqrt(2/n_per_grp)
delta_val  = effect_sz*sigma_exp
power_arr  = np.array([1-t_dist.cdf(t_dist.ppf(1-a/2,df=2*(n_per_grp-1)),
                                      df=2*(n_per_grp-1),
                                      nc=delta_val/se_diff) for a in alpha_arr])  # loop

# §5.1 Face embedding simulator
np.random.seed(42)
dim_embed = 128
n_identities = 50

# Simulate embeddings: each identity has a "true" vector + Gaussian noise
identity_centers = np.random.randn(n_identities, dim_embed)
identity_centers /= np.linalg.norm(identity_centers,axis=1,keepdims=True)

def get_embed(identity_id, noise=0.15):
    v = identity_centers[identity_id] + noise*np.random.randn(dim_embed)
    return v/np.linalg.norm(v)

def cosine_sim(a,b): return np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b))

# Generate gallery (1 img/identity) and probes (2 imgs/identity + impostors)
gallery = {i: get_embed(i,0.05) for i in range(n_identities)}  # loop: gallery

# Genuine scores (same identity)
n_gen = 500
genuine_scores = [cosine_sim(gallery[i%n_identities], get_embed(i%n_identities,0.2))
                  for i in range(n_gen)]  # loop: genuine

# Impostor scores (different identities)
n_imp = 500
import random; random.seed(2026)
impostor_scores = [cosine_sim(gallery[i%n_identities],
                               gallery[(i+17)%n_identities])
                   for i in range(n_imp)]  # loop: impostor

genuine_scores  = np.array(genuine_scores)
impostor_scores = np.array(impostor_scores)

# ROC curve
thresholds = np.linspace(0,1,200)
TAR = [np.mean(genuine_scores  >= th) for th in thresholds]  # loop: TAR
FAR = [np.mean(impostor_scores >= th) for th in thresholds]  # loop: FAR
TAR = np.array(TAR); FAR = np.array(FAR)

# EER (equal error rate)
FRR = 1-TAR
diffs = np.abs(FAR-FRR)
eer_idx = diffs.argmin()
EER = (FAR[eer_idx]+FRR[eer_idx])/2
print(f'\n§5.1 Face recognition (simulated {dim_embed}-D embeddings, {n_identities} identities):')
print(f'     Genuine mean cosine sim  = {genuine_scores.mean():.3f} ± {genuine_scores.std():.3f}')
print(f'     Impostor mean cosine sim = {impostor_scores.mean():.3f} ± {impostor_scores.std():.3f}')
print(f'     EER = {EER:.3f} @ threshold={thresholds[eer_idx]:.3f}')

# §5.2 Embedding space PCA
from numpy.linalg import eigh
all_embeds = np.vstack([gallery[i] for i in range(n_identities)])
cov_e = all_embeds.T @ all_embeds / n_identities
vals,vecs = eigh(cov_e)
idx_sort  = np.argsort(vals)[::-1]
PC1,PC2   = vecs[:,idx_sort[0]], vecs[:,idx_sort[1]]
proj_e    = all_embeds @ np.column_stack([PC1,PC2])
print(f'§5.2 PCA: top 2 PCs explain {(vals[idx_sort[:2]].sum()/vals.sum()*100):.1f}% variance')

# ── Plots ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(2,3,figsize=(16,9))

# Power vs effect size
axes[0][0].loglog(delta_over_sigma, n_required,'royalblue',lw=2.5)
axes[0][0].axhline(10000,color='green',ls='--',lw=2,label='N=10,000')
axes[0][0].axhline(50,color='orange',ls='--',lw=1.5,label='N=50')
axes[0][0].axhline(200,color='red',ls=':',lw=1.5,label='N=200')
axes[0][0].set_xlabel('Effect size δ/σ')
axes[0][0].set_ylabel('Required n per group')
axes[0][0].set_title(f'§4.1 Sample size for 80% power\n(α={alpha_level}, two-sample t-test)')
axes[0][0].legend(fontsize=8); axes[0][0].set_ylim(10,1e6)

# p-value distribution from simulation
axes[0][1].hist(p_values,bins=50,color='royalblue',edgecolor='white',alpha=0.8)
axes[0][1].axvline(alpha_level,color='red',lw=2,ls='--',label=f'α={alpha_level}')
reject_frac = np.mean(np.array(p_values)<alpha_level)
axes[0][1].set_xlabel('p-value'); axes[0][1].set_ylabel('Count (out of 10,000)')
axes[0][1].set_title(f'§4.2 10,000 MC experiments\n{reject_frac:.1%} rejected H₀ (= empirical power)')
axes[0][1].legend(fontsize=8)

# Alpha vs Power tradeoff
axes[0][2].plot(alpha_arr, power_arr,'royalblue',lw=2.5,label='Power(α)')
axes[0][2].plot(alpha_arr, alpha_arr,'red',lw=2,ls='--',label='Type I error (α)')
axes[0][2].fill_between(alpha_arr,alpha_arr,power_arr,alpha=0.15,color='green',
                         label='Power-α gap')
axes[0][2].set_xlabel('Significance level α')
axes[0][2].set_ylabel('Probability')
axes[0][2].set_title('§4.3 Power vs Type I error tradeoff\n(n=50/group, δ/σ=0.3)')
axes[0][2].legend(fontsize=8)

# Score distributions
axes[1][0].hist(genuine_scores,bins=40,alpha=0.7,color='green',label='Genuine',
                edgecolor='white',density=True)
axes[1][0].hist(impostor_scores,bins=40,alpha=0.7,color='red',label='Impostor',
                edgecolor='white',density=True)
axes[1][0].axvline(thresholds[eer_idx],color='navy',lw=2.5,ls='--',
                    label=f'EER threshold={thresholds[eer_idx]:.2f}')
axes[1][0].set_xlabel('Cosine similarity')
axes[1][0].set_ylabel('Density')
axes[1][0].set_title(f'§5.1 Face embedding score distributions\nEER={EER:.3f} ({dim_embed}-D embeddings)')
axes[1][0].legend(fontsize=8)

# ROC curve
axes[1][1].plot(FAR,TAR,'royalblue',lw=2.5,label='ROC')
axes[1][1].plot([0,1],[0,1],'gray',ls='--',lw=1,label='Random (AUC=0.5)')
axes[1][1].plot(FAR[eer_idx],TAR[eer_idx],'r*',ms=15,label=f'EER={EER:.3f}')
auc_roc = np.trapz(TAR[::-1],FAR[::-1])
axes[1][1].set_xlabel('FAR (False Accept Rate)')
axes[1][1].set_ylabel('TAR (True Accept Rate)')
axes[1][1].set_title(f'§5.1 ROC curve\nAUC={auc_roc:.3f}')
axes[1][1].legend(fontsize=8)

# PCA of embedding space
sc = axes[1][2].scatter(proj_e[:,0],proj_e[:,1],
                         c=np.arange(n_identities),cmap='tab20',s=80,zorder=3)
axes[1][2].set_xlabel('PC1'); axes[1][2].set_ylabel('PC2')
axes[1][2].set_title(f'§5.2 Face embedding space (PCA)\n{n_identities} identities in {dim_embed}-D → 2D')
plt.colorbar(sc,ax=axes[1][2],label='Identity ID')
plt.suptitle('§4: 10k Experiments · Power Analysis | §5: Face Recognition Pipeline',
             fontsize=12,fontweight='bold')
plt.tight_layout(); plt.show()

---
## §6 🌀 Golden Ratio + Symmetrical Buildings

$\varphi = \frac{1+\sqrt{5}}{2} \approx 1.618...$   — the unique ratio where $\varphi^2 = \varphi+1$.

**Fibonacci → golden ratio:** $F_{n+1}/F_n \to \varphi$

**In architecture:**
- Parthenon: facade width/height ≈ φ
- Le Corbusier's **Modulor**: human body proportions based on φ
- Cathedral nave: bay proportions, rib vault rise/span ≈ φ

**Symmetry groups in buildings:**
- Bilateral symmetry (D₁): most facades
- 4-fold symmetry (D₄): domes, Greek crosses
- Translational symmetry: colonnades, arches

**SymPy exact calculation:** $\varphi = \frac{1+\sqrt{5}}{2}$, minimal polynomial: $x^2-x-1=0$

In [ ]:
# §6 — Golden ratio + symmetrical buildings

# §6.1 Golden ratio: exact + algebraic properties
phi_sym = (1+sp.sqrt(5))/2
print('§6.1 Golden ratio:')
print(f'  φ = {phi_sym} ≈ {float(phi_sym):.10f}')
print(f'  φ² = {sp.expand(phi_sym**2)} = {sp.simplify(phi_sym**2-phi_sym-1)} + φ + 1')
print(f'  1/φ = {sp.simplify(1/phi_sym)} = {float(1/phi_sym):.10f}')
print(f'  φ - 1/φ = {sp.simplify(phi_sym-1/phi_sym)}')
display(Math(r'\varphi = \frac{1+\sqrt{5}}{2},\quad \varphi^2=\varphi+1,\quad \frac{1}{\varphi}=\varphi-1'))

phi = float(phi_sym)

# §6.2 Fibonacci convergence to phi
fib = [1,1]
for _ in range(30): fib.append(fib[-1]+fib[-2])  # loop: Fibonacci
ratios = [fib[i+1]/fib[i] for i in range(1,len(fib)-1)]
print(f'\n§6.2 Fibonacci ratios: {[f"{r:.6f}" for r in ratios[-5:]]}')
print(f'     Converges to φ={phi:.6f}')

# §6.3 Symmetrical building generator (matplotlib CAD)
def golden_rect(ax, x, y, w, depth=0, max_depth=5, color_idx=0):
    if depth > max_depth or w < 0.01: return
    h   = w/phi
    col = plt.cm.viridis(color_idx/max_depth)
    ax.add_patch(Rectangle((x,y),w,h,facecolor=col,edgecolor='black',alpha=0.6,lw=1))
    # Add the square
    ax.add_patch(Rectangle((x,y),h,h,facecolor='none',edgecolor='red',alpha=0.4,lw=0.7,ls='--'))
    golden_rect(ax,x+h,y,w-h,depth+1,max_depth,(color_idx+1)%6)

# §6.4 Symmetrical facade with golden proportions
def draw_facade(ax, W_facade=8.0, floors=4):
    H_floor  = W_facade/phi**2
    W_window = W_facade/(phi**3)
    H_window = W_window*phi
    ax.set_xlim(-0.5,W_facade+0.5); ax.set_ylim(-0.5,H_floor*floors+1.5)
    ax.set_aspect('equal'); ax.axis('off')
    ax.set_title('§6 Symmetrical facade\n(golden ratio proportions)', fontsize=10)

    # Foundation
    ax.add_patch(Rectangle((-0.3,-0.5),W_facade+0.6,0.5,facecolor='#5d4037',
                            edgecolor='#3e2723',lw=2))
    # Walls
    ax.add_patch(Rectangle((0,0),W_facade,H_floor*floors,facecolor='#f5f5dc',
                            edgecolor='#555',lw=2))
    # Cornice
    ax.add_patch(Rectangle((-0.2,H_floor*floors),W_facade+0.4,0.25,
                            facecolor='#9e9e9e',edgecolor='#555',lw=1.5))
    # Roof
    roof_x = np.array([-0.5,W_facade/2,W_facade+0.5])
    roof_y = np.array([H_floor*floors+0.25,H_floor*floors+0.25+W_facade/phi**2*0.7,
                        H_floor*floors+0.25])
    ax.fill(roof_x,roof_y,facecolor='#b71c1c',edgecolor='#7f0000',lw=2)

    # Windows (symmetric grid)
    n_win = 5   # odd for symmetry
    x_centers = np.linspace(W_facade/2-(n_win//2)*W_facade/n_win,
                             W_facade/2+(n_win//2)*W_facade/n_win, n_win)
    for floor_i in range(floors):           # loop: floors
        for xc in x_centers:               # loop: windows per floor
            wx = xc - W_window/2; wy = floor_i*H_floor + (H_floor-H_window)/2
            ax.add_patch(Rectangle((wx,wy),W_window,H_window,
                                    facecolor='#b3e5fc',edgecolor='#555',lw=1.5))
            # Window divider
            ax.plot([wx+W_window/2,wx+W_window/2],[wy,wy+H_window],'#555',lw=0.8)
            ax.plot([wx,wx+W_window],[wy+H_window/2,wy+H_window/2],'#555',lw=0.8)

    # Door (center, ground floor, golden rect)
    dw = W_window*phi*0.7; dh = dw*phi
    ax.add_patch(Rectangle((W_facade/2-dw/2,0),dw,dh,facecolor='#795548',
                            edgecolor='#3e2723',lw=2))
    ax.add_patch(mpatches.Ellipse((W_facade/2,dh),dw,dw*0.4,
                                   facecolor='#f5f5dc',edgecolor='#555',lw=1.5))
    # Columns flanking door
    for cx in [W_facade/2-dw*0.9, W_facade/2+dw*0.9]:    # loop: columns
        ax.add_patch(Rectangle((cx-0.1,0),0.2,dh*1.2,facecolor='#e0e0e0',edgecolor='#555',lw=1.5))

    # Dimension annotation
    ax.annotate('',xy=(W_facade,H_floor*floors),xytext=(0,H_floor*floors),
                arrowprops=dict(arrowstyle='<->',color='blue',lw=1.5))
    ax.text(W_facade/2,H_floor*floors+0.5,f'W={W_facade:.1f}m',ha='center',
            fontsize=9,color='blue')
    ax.annotate('',xy=(W_facade+0.4,H_floor*floors),xytext=(W_facade+0.4,0),
                arrowprops=dict(arrowstyle='<->',color='green',lw=1.5))
    ax.text(W_facade+0.3,H_floor*floors/2,f'H={H_floor*floors:.1f}m\nW/H={W_facade/(H_floor*floors):.2f}',
            ha='left',fontsize=8,color='green')

# ── Plots ─────────────────────────────────────────────────────────
fig = plt.figure(figsize=(16,10))
gs  = gridspec.GridSpec(2,3,fig,hspace=0.35,wspace=0.3)

# Golden spiral / rectangle recursion
ax_gs = fig.add_subplot(gs[0,0])
ax_gs.set_xlim(0,3.5); ax_gs.set_ylim(0,2.2); ax_gs.set_aspect('equal')
golden_rect(ax_gs, 0, 0, phi**3, max_depth=7)
# Spiral overlay
theta_sp = np.linspace(0,4*np.pi,2000)
r_sp     = phi**(2*theta_sp/np.pi)
x_sp     = r_sp*np.cos(theta_sp)*0.05+1.8; y_sp=r_sp*np.sin(theta_sp)*0.05+0.5
ax_gs.plot(x_sp,y_sp,'white',lw=1.5,alpha=0.7)
ax_gs.set_title(f'§6.1 Golden rectangle recursion\nφ={phi:.4f}, each rect = φ×1')
ax_gs.axis('off')

# Fibonacci convergence
ax_fib = fig.add_subplot(gs[0,1])
ax_fib.plot(range(2,len(ratios)+2), ratios, 'royalblue', lw=2.5, marker='o', ms=5)
ax_fib.axhline(phi, color='red', lw=2, ls='--', label=f'φ={phi:.6f}')
ax_fib.set_xlabel('Fibonacci index n')
ax_fib.set_ylabel('F(n+1)/F(n)')
ax_fib.set_title('§6.2 Fibonacci ratios → φ\nexponential convergence')
ax_fib.legend(fontsize=9)

# Phi powers table
ax_tab = fig.add_subplot(gs[0,2])
ax_tab.axis('off')
powers_data = [(n, phi**n, sp.simplify(phi_sym**n)) for n in range(-3,8)]
table_str = 'n    φⁿ (decimal)  SymPy exact\n' + '-'*42 + '\n'
for n,pn,sp_n in powers_data:
    table_str += f'{n:3d}  {pn:12.6f}  {sp_n}\n'
ax_tab.text(0.05,0.95,table_str,transform=ax_tab.transAxes,
            fontsize=8.5,verticalalignment='top',fontfamily='monospace',
            bbox=dict(boxstyle='round',facecolor='#f0f8ff',alpha=0.9))
ax_tab.set_title('§6.1 Powers of φ\n(note: φⁿ = φⁿ⁻¹+φⁿ⁻²)')

# Symmetrical facade
ax_fac = fig.add_subplot(gs[1,:2])
ax_fac.set_facecolor('#87ceeb')   # sky
draw_facade(ax_fac, W_facade=8.0, floors=4)

# Phi vs natural constants
ax_phi = fig.add_subplot(gs[1,2])
constants = {
    'φ (golden)': phi,
    'e (Euler)':  np.e,
    'π/2':        np.pi/2,
    'φ²':         phi**2,
    '√2':         np.sqrt(2),
    '√3':         np.sqrt(3),
    '4/π':        4/np.pi,
}
names_c = list(constants.keys()); vals_c = list(constants.values())
bars_c  = ax_phi.barh(range(len(names_c)),vals_c,
                       color=plt.cm.RdYlGn(np.linspace(0.1,0.9,len(vals_c))),
                       edgecolor='#333',alpha=0.85)
ax_phi.set_yticks(range(len(names_c))); ax_phi.set_yticklabels(names_c)
ax_phi.set_xlabel('Value'); ax_phi.set_title('§6 φ among famous constants\n(near π/2, e — coincidence?)')
ax_phi.axvline(phi,color='red',lw=2,ls='--',alpha=0.5)
for bar,v in zip(bars_c,vals_c):   # loop: label bars
    ax_phi.text(v+0.02,bar.get_y()+bar.get_height()/2,f'{v:.4f}',
                va='center',fontsize=8)
plt.suptitle('§6: Golden Ratio φ · Fibonacci · Symmetrical Buildings · Architectural Proportions',
             fontsize=12,fontweight='bold')
plt.tight_layout(); plt.show()

---
## §7 🔧 POSIX Makefile for Python Projects

**Why Makefile?** Dependency tracking — targets only rebuild when sources change.
`make` is a DAG executor: leaf targets → derived targets → final output.

**POSIX compliance:** use `$(PYTHON)`, `$(RM)`, `/bin/sh`, avoid bash-isms.

**Phony targets** (always run, no file output): `.PHONY: all install test lint clean`

```makefile
PYTHON  := python3
VENV    := .venv
PIP     := $(VENV)/bin/pip
PYTEST  := $(VENV)/bin/pytest
```

**Standard Python project Makefile targets:**
- `make install` — create venv, install deps
- `make test` — run pytest with coverage
- `make lint` — ruff + mypy type check
- `make docs` — sphinx build
- `make clean` — remove build artifacts
- `make dist` — build wheel + sdist

In [ ]:
# §7 — POSIX Makefile for Python + project structure generator

makefile_src = '''# ════════════════════════════════════════════════
# Makefile — POSIX-compliant Python project build
# Usage: make install && make test && make lint
# ════════════════════════════════════════════════
SHELL    := /bin/sh
PYTHON   := python3
VENV     := .venv
BIN      := $(VENV)/bin
PIP      := $(BIN)/pip
PYTEST   := $(BIN)/pytest
RUFF     := $(BIN)/ruff
MYPY     := $(BIN)/mypy
SPHINX   := $(BIN)/sphinx-build
TWINE    := $(BIN)/twine

PKG_NAME := mypackage
SRC_DIR  := src/$(PKG_NAME)
TEST_DIR := tests
DOCS_DIR := docs
DIST_DIR := dist

.PHONY: all install dev test lint typecheck docs clean dist upload help

# ── Default target ──────────────────────────────────────────────
all: install lint test

# ── Environment ─────────────────────────────────────────────────
$(VENV)/.done: requirements.txt
\t$(PYTHON) -m venv $(VENV)
\t$(PIP) install --upgrade pip
\t$(PIP) install -r requirements.txt
\ttouch $(VENV)/.done

install: $(VENV)/.done   ## Create venv and install runtime deps

dev: install              ## Also install dev deps
\t$(PIP) install -r requirements-dev.txt

# ── Tests ───────────────────────────────────────────────────────
test: install             ## Run pytest with coverage
\t$(PYTEST) $(TEST_DIR) \\
\t  --cov=$(SRC_DIR) \\
\t  --cov-report=term-missing \\
\t  --cov-report=html:htmlcov \\
\t  -v

test-fast: install        ## Run tests without coverage (faster)
\t$(PYTEST) $(TEST_DIR) -x -q

# ── Code quality ────────────────────────────────────────────────
lint: dev                 ## Ruff linter (replaces flake8+isort+pyupgrade)
\t$(RUFF) check $(SRC_DIR) $(TEST_DIR)
\t$(RUFF) format --check $(SRC_DIR) $(TEST_DIR)

lint-fix: dev             ## Auto-fix lint issues
\t$(RUFF) check --fix $(SRC_DIR) $(TEST_DIR)
\t$(RUFF) format $(SRC_DIR) $(TEST_DIR)

typecheck: dev            ## mypy static type checking
\t$(MYPY) $(SRC_DIR) --ignore-missing-imports

# ── Docs ────────────────────────────────────────────────────────
docs: dev                 ## Build Sphinx HTML docs
\t$(SPHINX) -b html $(DOCS_DIR)/source $(DOCS_DIR)/_build

# ── Distribution ────────────────────────────────────────────────
dist: clean install       ## Build wheel + sdist
\t$(PYTHON) -m build --outdir $(DIST_DIR)

upload: dist              ## Upload to PyPI via twine
\t$(TWINE) upload $(DIST_DIR)/*

upload-test: dist         ## Upload to TestPyPI first
\t$(TWINE) upload --repository testpypi $(DIST_DIR)/*

# ── Cleanup ─────────────────────────────────────────────────────
clean:                    ## Remove all build artifacts
\t$(RM) -rf $(DIST_DIR) build htmlcov .coverage
\t$(RM) -rf $(VENV)
\tfind . -type d -name __pycache__ -exec $(RM) -rf {} + 2>/dev/null || true
\tfind . -type f -name "*.pyc" -delete 2>/dev/null || true

# ── Help ────────────────────────────────────────────────────────
help:                     ## Print this help
\t@grep -E "^[a-zA-Z_-]+:.*##" $(MAKEFILE_LIST) | \\
\t awk "BEGIN{FS=\":.*##\"}{printf \"  %-15s %s\\n\",$$1,$$2}"
'''

# Project structure (also generate a scaffold generator)
project_scaffold = {
    'src/mypackage/__init__.py': "'''My package.'''\n__version__ = '0.1.0'\n",
    'src/mypackage/core.py':     "'''Core module.'''\nfrom typing import Any\n\ndef hello(name: str) -> str:\n    return f'Hello, {name}!'\n",
    'tests/__init__.py':         '',
    'tests/test_core.py':        "'''Tests for core.'''\nimport pytest\nfrom mypackage.core import hello\n\ndef test_hello():\n    assert hello('World') == 'Hello, World!'\n",
    'requirements.txt':          'numpy>=1.24\nscipy>=1.10\nmatplotlib>=3.7\n',
    'requirements-dev.txt':      '-r requirements.txt\npytest>=7.4\npytest-cov>=4.1\nruff>=0.1\nmypy>=1.5\nsphinx>=7.0\nbuild>=1.0\ntwine>=4.0\n',
    'pyproject.toml':            '[build-system]\nrequires = ["setuptools>=68","wheel"]\nbuild-backend = "setuptools.backends.legacy:build"\n\n[project]\nname = "mypackage"\nversion = "0.1.0"\nrequires-python = ">=3.11"\n',
}

print('§7 POSIX Makefile for Python:')
print('─'*60)
# Show key sections
for line in makefile_src.split('\n')[5:35]:
    print(line)
print('...')
print('\n§7 Project scaffold files:')
for path,content in project_scaffold.items():    # loop: scaffold
    print(f'  {path} ({len(content)} bytes)')

# §7.2 Make dependency graph
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1,2,figsize=(16,7))

# Makefile content display
axes[0].axis('off')
axes[0].text(0.01,0.99,makefile_src[:1400],transform=axes[0].transAxes,
             fontsize=6.5,verticalalignment='top',fontfamily='monospace',
             bbox=dict(boxstyle='square',facecolor='#1e1e1e',alpha=0.97))
axes[0].set_facecolor('#1e1e1e')
axes[0].set_title('§7 POSIX Makefile (Python project)\nSyntax-highlighted view', color='white')
axes[0].title.set_color('#ffffff')
axes[0].set_facecolor('#1e1e1e')

# Make DAG visualization
ax_dag = axes[1]
ax_dag.set_xlim(0,10); ax_dag.set_ylim(0,8)
ax_dag.axis('off')
ax_dag.set_title('§7 Makefile dependency DAG\n(targets + dependencies)', fontsize=11)

targets = {
    'all':       (5,7.5,'#1565c0'),
    'install':   (2,6.0,'#2e7d32'),
    'dev':       (5,6.0,'#1b5e20'),
    'test':      (1,4.5,'#e65100'),
    'lint':      (4,4.5,'#bf360c'),
    'typecheck': (7,4.5,'#880e4f'),
    'docs':      (9,4.5,'#4a148c'),
    'dist':      (3,3.0,'#33691e'),
    'upload':    (3,1.5,'#1a237e'),
    'clean':     (7,3.0,'#b71c1c'),
    'venv/.done':(2,4.5,'#004d40'),
    'requirements.txt':(0.5,3.0,'#37474f'),
}

dep_edges = [
    ('all','install'),('all','lint'),('all','test'),
    ('install','venv/.done'),('venv/.done','requirements.txt'),
    ('dev','install'),('test','install'),('lint','dev'),
    ('typecheck','dev'),('docs','dev'),('dist','install'),
    ('dist','clean'),('upload','dist'),
]

# Draw nodes
for name,(x,y,col) in targets.items():    # loop: draw nodes
    ax_dag.add_patch(FancyBboxPatch((x-0.9,y-0.25),1.8,0.5,
                                     boxstyle='round,pad=0.05',
                                     facecolor=col,edgecolor='white',lw=1.5,alpha=0.9))
    ax_dag.text(x,y,name,ha='center',va='center',fontsize=7.5,
                color='white',fontweight='bold')

# Draw edges
for (src,dst) in dep_edges:           # loop: draw edges
    xs,ys = targets[src][:2]; xd,yd = targets[dst][:2]
    ax_dag.annotate('',xy=(xd,yd+0.25),xytext=(xs,ys-0.25),
                    arrowprops=dict(arrowstyle='-|>',color='#aaa',
                                    lw=1.5,connectionstyle='arc3,rad=0.1'))

ax_dag.text(5,0.5,'make all = install → lint → test\n(full CI pipeline in one command)',
            ha='center',fontsize=9,color='navy',
            bbox=dict(boxstyle='round',facecolor='#e3f2fd',alpha=0.9))
plt.suptitle('§7: POSIX Makefile — Python Project Build System',
             fontsize=12,fontweight='bold')
plt.tight_layout(); plt.show()

# §7.3 POSIX tools table
posix_tools = [
    ('make',   'Build automation, dependency tracking'),
    ('ar',     'Create static libraries (.a archives)'),
    ('nm',     'List symbols in object files'),
    ('strip',  'Remove debug symbols from binary'),
    ('ldd',    'List shared library dependencies'),
    ('strace', 'Trace system calls (Linux)'),
    ('valgrind','Memory error detector, profiler'),
    ('objdump','Disassemble object files'),
    ('readelf','ELF binary inspector'),
    ('pkg-config','Query installed library flags'),
]
print('\n§7 POSIX/Linux build tools for Python C extensions:')
for tool,desc in posix_tools:    # loop: print table
    print(f'  {tool:<12} {desc}')